In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

df_dev = pd.read_csv('development.csv').fillna('')
df_eval = pd.read_csv('evaluation.csv').fillna('')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_dev['final_text'] = (df_dev['title'] + " " + df_dev['article']).apply(clean_text)
df_eval['final_text'] = (df_eval['title'] + " " + df_eval['article']).apply(clean_text)

label_counts = df_dev.groupby(['final_text', 'label']).size().reset_index(name='count')
label_counts = label_counts.sort_values(['final_text', 'count'], ascending=[True, False])
best_labels = label_counts.drop_duplicates(subset=['final_text'], keep='first')

meta_data = df_dev.drop_duplicates(subset=['final_text']).set_index('final_text')
df_clean = best_labels[['final_text', 'label']].join(meta_data[['source', 'page_rank']], on='final_text')

In [10]:
df_dev

,Id,source,title,article,page_rank,timestamp,label,final_text
0,0,AllAfrica.com,OPEC Boosts Nigeria&#39;s Oil Revenue By .82m Bpd,THE Organisation of Petroleum Exporting Countr...,5,2004-09-16 22:39:53,5,opec boosts nigeria&#;s oil revenue by .m bpd ...
1,1,Xinhua,Yearender: Mideast peace roadmap reaches dead-...,Looking back at the major events that took pla...,5,2004-12-17 19:01:14,0,yearender: mideast peace roadmap reaches dead-...
2,2,Yahoo,Battleground Dispatches for Oct. 5 \\n (CQP...,CQPolitics.com - Here are today's Battleground...,5,2006-10-05 18:42:29,0,battleground dispatches for oct. \ (cqpolitics...
3,3,BBC,Air best to resuscitate newborns,Air rather than oxygen should be used to resus...,5,0000-00-00 00:00:00,0,air best to resuscitate newborns air rather th...
4,4,Yahoo,High tech German train crash kills at least on...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2006-09-22 17:28:57,0,high tech german train crash kills at least on...
...,...,...,...,...,...,...,...,...
79992,79992,Yahoo,Italy's embattled Prodi faces vote of confiden...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2008-01-23 11:39:35,0,italy's embattled prodi faces vote of confiden...
79993,79993,All-Baseball.com,"Ding Dong, the Deal is Dead","As yesterday began, there was widespread antic...",5,0000-00-00 00:00:00,4,"ding dong, the deal is dead as yesterday began..."
79994,79994,Yahoo,Two bombs discovered in Sardinia after Berlusc...,AFP - Police discovered two bombs near the Sar...,5,0000-00-00 00:00:00,0,two bombs discovered in sardinia after berlusc...
79995,79995,Voice,Red Cross Report Alleges US Detainee Abuse at ...,A report by the International Committee of the...,5,0000-00-00 00:00:00,3,red cross report alleges us detainee abuse at ...


In [12]:
X = df_clean
y = df_clean['label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

word_tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    sublinear_tf=True
    )

char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    sublinear_tf=True
    )

text_features = FeatureUnion([
    ('word', word_tfidf),
    ('char', char_tfidf)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_features, 'final_text'),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['source']),
        ('num', StandardScaler(), ['page_rank'])
    ],
    remainder='drop'
)


In [13]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', n_jobs=-1))
])

print("Training Logistic Regression model...")
pipeline.fit(X_train, y_train)

Training Logistic Regression model...


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  FeatureUnion(transformer_list=[('word',
                                                                                  TfidfVectorizer(ngram_range=(1,
                                                                                                               2),
                                                                                                  sublinear_tf=True)),
                                                                                 ('char',
                                                                                  TfidfVectorizer(analyzer='char_wb',
                                                                                                  ngram_range=(3,
                                                                                                               5),
                                                                                                  sublinear_tf=True))]),
                                                  'final_text'),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['source']),
                                                 ('num', StandardScaler(),
                                                  ['page_rank'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    n_jobs=-1))])

In [14]:
print("\nValidation Report:")
val_preds = pipeline.predict(X_val)
print(classification_report(y_val, val_preds))
print(f"Macro F1 Score: {f1_score(y_val, val_preds, average='macro'):.4f}")



Validation Report:
              precision    recall  f1-score   support

           0       0.79      0.69      0.74      4613
           1       0.75      0.81      0.78      2068
           2       0.83      0.82      0.82      2161
           3       0.56      0.56      0.56      1892
           4       0.81      0.92      0.86      1684
           5       0.52      0.52      0.52      2355
           6       0.59      0.82      0.69       570

    accuracy                           0.71     15343
   macro avg       0.69      0.73      0.71     15343
weighted avg       0.71      0.71      0.71     15343

Macro F1 Score: 0.7093


In [15]:
print("\nRetraining on full dataset...")
pipeline.fit(X, y)

print("Generating submission file...")
test_preds = pipeline.predict(df_eval)

submission = pd.DataFrame({'Id': df_eval['Id'], 'Predicted': test_preds})
submission.to_csv('submission_logistic_regression.csv', index=False)
print("Done. File saved as 'submission_logistic_regression.csv'.")


Retraining on full dataset...
Generating submission file...
Done. File saved as 'submission_logistic_regression.csv'.
